# 00 - Workflow Lengkap Structural Damage Classification

**Mahasiswa:** Rahardian Putra  
**NRP:** 5052231018

**Kompetisi:** [Structural Damage Classification](https://www.kaggle.com/competitions/structural-damage-classification)
**Metrik resmi:** Macro F1-Score · **Target:** `damage_grade` (`Grade 1` … `Grade 5`)
**Batasan tugas:** tanpa Neural Network, wajib memakai `Pipeline` + `ColumnTransformer`.

---

### Konteks

Gempa Gorkha, Nepal (25 April 2015, Mw 7,8) merusak ratusan ribu bangunan. Central Bureau of
Statistics (CBS) Nepal mendata karakteristik struktural bangunan **sebelum** gempa (tipe fondasi,
material dinding, konfigurasi atap, umur, kondisi tanah) beserta **tingkat kerusakan aktual**
setelah gempa. Tugasnya: memprediksi tingkat kerusakan dari fitur pra-gempa.

### Alur notebook

| Bagian | Isi |
|---|---|
| 1 | Setup, konfigurasi, pencarian lokasi data |
| 2 | Memuat data dan audit kualitas (skema, nilai hilang, duplikasi, kebocoran ID) |
| 3 | EDA: ketidakseimbangan kelas, fitur numerik, fitur kategorikal, **sinyal geografis** |
| 4 | Feature engineering + `ColumnTransformer` (bebas kebocoran) |
| 5 | Protokol validasi *stratified* 80/20 |
| 6 | Baseline: Dummy, Logistic Regression, Random Forest |
| 7 | Penyetelan LightGBM dengan `RandomizedSearchCV` (hanya pada *development*) |
| 8 | Evaluasi *holdout*: classification report, confusion matrix, permutation importance |
| 9 | Pelatihan final pada seluruh data latih + pembuatan `submission.csv` |
| 10 | Kesimpulan dan keterbatasan |

> **Catatan penting 1 — format label submission.** Halaman `Overview` meminta kategori
> **1–5**, dan contoh submission-nya juga angka, tetapi `train.csv` serta `sample_submission.csv`
> memakai string `Grade 1`–`Grade 5`. Skrip penilai membaca `damage_grade` sebagai bilangan bulat,
> sehingga mengirim string menghasilkan
> *`Evaluation Exception: The input string 'Grade' was not in a correct format`*.
> Karena itu **label internal model tetap string** (agar classification report per kelas mudah
> dibaca), sedangkan **berkas submission ditulis sebagai bilangan bulat 1–5** melalui
> `SUBMISSION_LABEL_STYLE = "numeric"`. Salinan bergaya `Grade N` tetap dibuat sebagai
> `submission_grade_labels.csv`.
>
> **Catatan penting 2 — metadata vs CSV.** `metadata.csv` mendeskripsikan 11 *flag* material
> (mis. `has_superstructure_adobe_mud`), sedangkan CSV aktual memuat satu kolom kategorikal
> `superstructure`. Kode mengikuti CSV aktual, bukan metadata.

## 1. Setup

Seluruh dependensi di bawah ini sudah tersedia pada image Kaggle (CPU) dan tidak memerlukan
instalasi tambahan.

In [ ]:
from __future__ import annotations

import json
import platform
import time
import warnings
from pathlib import Path

import joblib
import lightgbm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Peringatan ini bersifat kosmetik: LightGBM menerima DataFrame saat fit lalu
# numpy saat predict di dalam cross-validation.
warnings.filterwarnings("ignore", message="X does not have valid feature names")

SEED = 42
LABELS = ["Grade 1", "Grade 2", "Grade 3", "Grade 4", "Grade 5"]

# Pemetaan label internal (string, sama seperti train.csv) ke kode numerik 1-5.
LABEL_TO_CODE = {label: index + 1 for index, label in enumerate(LABELS)}
CODE_TO_LABEL = {code: label for label, code in LABEL_TO_CODE.items()}

# Gaya label pada berkas submission.
#   "numeric" -> damage_grade berisi 1..5
#   "grade"   -> damage_grade berisi "Grade 1".."Grade 5" (seperti sample_submission.csv)
# Halaman Overview dan contoh submission memakai angka 1-5, dan skrip penilai
# membaca kolom ini sebagai bilangan bulat: mengirim "Grade 1" menghasilkan
# "Evaluation Exception: The input string 'Grade' was not in a correct format."
# Karena itu submission memakai "numeric", sedangkan label internal model tetap
# string agar classification report per kelas mudah dibaca.
SUBMISSION_LABEL_STYLE = "numeric"

# QUICK_MODE hanya untuk menguji alur dengan cepat. Hasil QUICK_MODE bukan hasil
# kompetisi dan submission-nya disimpan dengan nama berkas berbeda.
QUICK_MODE = False
FULL_REFIT = True

# Model final dilatih beberapa kali dengan seed berbeda, lalu probabilitasnya
# dirata-ratakan. Rata-rata seed mengurangi varians prediksi tanpa mengubah
# arsitektur model. QUICK_MODE memakai satu seed agar uji cepat tetap singkat.
FINAL_SEEDS = [SEED] if QUICK_MODE else [SEED, SEED + 137, SEED + 911]

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "figure.autolayout": True})

print("Python   :", platform.python_version())
print("pandas   :", pd.__version__, "| numpy   :", np.__version__)
print("scikit   :", sklearn.__version__, "| lightgbm:", lightgbm.__version__)

In [ ]:
def find_data_dir() -> Path:
    """Cari folder yang memuat ``train.csv`` dan ``test.csv``.

    Mendukung struktur input kompetisi Kaggle maupun folder ``data/`` lokal,
    sehingga notebook dapat dijalankan di dua lingkungan tanpa diubah.
    """
    candidates = [
        Path("/kaggle/input/structural-damage-classification"),
        Path("/kaggle/input/competitions/structural-damage-classification"),
        Path.cwd() / "data",
        Path.cwd(),
        Path.cwd().parent / "data",
    ]
    for folder in candidates:
        if (folder / "train.csv").exists() and (folder / "test.csv").exists():
            return folder
    for train_path in sorted(Path("/kaggle/input").glob("**/train.csv")):
        if (train_path.parent / "test.csv").exists():
            return train_path.parent
    raise FileNotFoundError("train.csv/test.csv tidak ditemukan; atur DATA_DIR secara manual.")


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path.cwd()
FIG_DIR = OUTPUT_DIR / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR   :", DATA_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("Berkas     :", sorted(p.name for p in DATA_DIR.glob("*.csv")))

## 2. Memuat data dan audit kualitas

Audit dilakukan sebelum pemodelan supaya asumsi protokol validasi dapat dipertanggungjawabkan,
khususnya: apakah `building_id` unik, apakah ada kebocoran antara train dan test, dan apakah ada
nilai hilang yang perlu ditangani.

In [ ]:
def load_data(data_dir: Path):
    """Muat train, test, dan sample submission sebagai DataFrame."""
    train = pd.read_csv(data_dir / "train.csv")
    test = pd.read_csv(data_dir / "test.csv")
    sample = pd.read_csv(data_dir / "sample_submission.csv")
    return train, test, sample


train, test, sample = load_data(DATA_DIR)
print("train :", train.shape, "| test :", test.shape, "| sample :", sample.shape)
display(train.head())

In [ ]:
def audit_data(train: pd.DataFrame, test: pd.DataFrame, sample: pd.DataFrame) -> pd.DataFrame:
    """Periksa skema, label, nilai hilang, duplikasi, dan tumpang tindih ID.

    Mengembalikan tabel pemeriksaan agar hasilnya dapat dibaca sebagai laporan,
    bukan sekadar lolos tanpa error.
    """
    checks = [
        ("Baris train", len(train)),
        ("Baris test", len(test)),
        ("Kolom train", train.shape[1]),
        ("Kolom test", test.shape[1]),
        ("Label target unik", ", ".join(sorted(train.damage_grade.unique()))),
        ("Nilai hilang train", int(train.isna().sum().sum())),
        ("Nilai hilang test", int(test.isna().sum().sum())),
        ("building_id train unik", bool(train.building_id.is_unique)),
        ("building_id test unik", bool(test.building_id.is_unique)),
        ("Irisan ID train dan test", len(set(train.building_id) & set(test.building_id))),
        ("Baris train duplikat penuh", int(train.duplicated().sum())),
        ("Fitur identik (tanpa ID/target)",
         int(train.drop(columns=["building_id", "damage_grade"]).duplicated().sum())),
        ("Kolom fitur train sama dengan test",
         set(train.columns) - {"damage_grade"} == set(test.columns)),
        ("Urutan ID sample sama dengan test", bool(sample.building_id.equals(test.building_id))),
    ]
    return pd.DataFrame(checks, columns=["Pemeriksaan", "Nilai"])


audit = audit_data(train, test, sample)
display(audit)

# Gagalkan lebih awal bila asumsi dasar protokol validasi tidak terpenuhi.
assert sorted(train.damage_grade.unique()) == LABELS, "Label target tidak sesuai dugaan."
assert set(train.building_id).isdisjoint(test.building_id), "Ada kebocoran building_id."
assert list(sample.columns) == ["building_id", "damage_grade"], "Skema sample tidak sesuai."
print("Seluruh pemeriksaan struktur data lolos.")

## 3. Analisis data eksploratif

### 3.1 Ketidakseimbangan kelas

Macro F1 memberi bobot **sama** untuk setiap kelas, sehingga kelas minoritas (Grade 1–2) sama
pentingnya dengan Grade 5. Karena itu akurasi bukan metrik yang tepat, dan *class weight* perlu
diuji.

In [ ]:
counts = train.damage_grade.value_counts().reindex(LABELS)
class_table = pd.DataFrame({"Jumlah": counts,
                            "Persentase (%)": (counts / counts.sum() * 100).round(2)})
display(class_table)
print(f"Rasio kelas terbanyak terhadap terkecil = {counts.max() / counts.min():.2f} : 1")

fig, ax = plt.subplots(figsize=(6, 3.5))
counts.plot.bar(ax=ax, color="#4C72B0", rot=0)
ax.set_title("Distribusi kelas damage_grade pada data latih")
ax.set_xlabel("Tingkat kerusakan")
ax.set_ylabel("Jumlah bangunan")
for position, value in enumerate(counts):
    ax.text(position, value, f"{value / 1000:.0f}k", ha="center", va="bottom", fontsize=9)
fig.savefig(FIG_DIR / "01_01_distribusi_kelas_v1.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Fitur numerik

Empat fitur numerik asli: jumlah lantai, umur bangunan, luas tapak, dan tinggi bangunan.
Perhatikan `age_building = 999`, yang hampir pasti merupakan kode "tidak diketahui" dan bukan umur
999 tahun. Nilainya **tidak** diganti secara spekulatif; sebagai gantinya ditambahkan indikator
biner pada tahap feature engineering.

In [ ]:
NUMERIC_RAW = ["count_floors_pre_eq", "age_building", "plinth_area_sq_ft", "height_ft_pre_eq"]

display(train[NUMERIC_RAW].describe().T.round(2))
print("Baris dengan age_building = 999 :", int(train.age_building.eq(999).sum()),
      f"({train.age_building.eq(999).mean():.2%})")

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, column in zip(axes, NUMERIC_RAW):
    upper = train[column].quantile(0.99)
    grouped = [train.loc[train.damage_grade == label, column].clip(upper=upper) for label in LABELS]
    ax.boxplot(grouped, showfliers=False)
    ax.set_xticklabels([label.replace("Grade ", "G") for label in LABELS])
    ax.set_title(column, fontsize=10)
    ax.set_xlabel("")
fig.suptitle("Sebaran fitur numerik menurut tingkat kerusakan (dipotong pada persentil 99)")
fig.savefig(FIG_DIR / "02_numeric_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.3 Fitur kategorikal

Sebelas kolom kategorikal. Tiga di antaranya bersifat geografis dan berhirarki
(`ward_id` ⊂ `vdcmun_id` ⊂ `district_id`), sedangkan `ward_id` ber-kardinalitas tinggi.

In [ ]:
GEO_COLS = ["district_id", "vdcmun_id", "ward_id"]
OTHER_CAT_COLS = ["land_surface_condition", "foundation_type", "roof_type", "ground_floor_type",
                  "other_floor_type", "position", "plan_configuration", "superstructure"]
CAT_COLS = GEO_COLS + OTHER_CAT_COLS


def categorical_profile(frame: pd.DataFrame, column: str,
                        target: str = "damage_grade") -> pd.DataFrame:
    """Hitung proporsi tiap tingkat kerusakan di dalam setiap kategori."""
    table = pd.crosstab(frame[column], frame[target], normalize="index")
    table = table.reindex(columns=LABELS, fill_value=0.0)
    return table.loc[frame[column].value_counts().index]


cardinality = pd.DataFrame({
    "n_kategori": [train[column].nunique() for column in CAT_COLS],
    "pangsa_kategori_terbesar (%)": [
        round(train[column].value_counts(normalize=True).iloc[0] * 100, 2) for column in CAT_COLS],
}, index=CAT_COLS).sort_values("n_kategori", ascending=False)
display(cardinality)

# Apakah semua kategori pada test juga muncul pada train?
unseen = pd.Series({column: len(set(test[column]) - set(train[column])) for column in CAT_COLS})
print("Kategori pada test yang tidak ada di train:", int(unseen.sum()), "buah")
display(unseen.to_frame("jumlah kategori baru"))

plot_cols = ["foundation_type", "superstructure", "roof_type",
             "ground_floor_type", "position", "plan_configuration"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, column in zip(axes.ravel(), plot_cols):
    categorical_profile(train, column).plot.barh(stacked=True, ax=ax, width=0.8)
    ax.set_title(column, fontsize=10)
    ax.set_xlabel("proporsi")
    ax.set_ylabel("")
    ax.invert_yaxis()
handles, legend_labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="lower center", ncol=5, frameon=False)
fig.suptitle("Proporsi tingkat kerusakan menurut fitur kategorikal", y=0.99)
fig.savefig(FIG_DIR / "03_categorical_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4 Sinyal geografis — temuan terpenting

Sebelum melatih model apa pun, mari ukur kekuatan sinyal lokasi dengan aturan paling sederhana:
**prediksi kelas mayoritas di dalam wilayah**. Jika aturan naif ini sudah tinggi, artinya kerusakan
terkonsentrasi secara spasial dan lokasi adalah fitur dominan.

Tabel di bawah memakai pembagian *development/holdout* yang sama dengan protokol pemodelan nanti,
sehingga angkanya sebanding.

In [ ]:
# Pembagian yang sama persis dengan protokol pemodelan, agar angkanya sebanding.
split_index = np.arange(len(train))
idx_dev, idx_hold = train_test_split(split_index, test_size=0.2,
                                     stratify=train.damage_grade, random_state=SEED)

geo_rows = []
for key in GEO_COLS:
    majority = pd.crosstab(train.loc[idx_dev, key],
                           train.loc[idx_dev, "damage_grade"]).idxmax(axis=1)
    prediction = train.loc[idx_hold, key].map(majority).fillna(train.damage_grade.mode()[0])
    geo_rows.append({"wilayah": key,
                     "n_kategori": int(train[key].nunique()),
                     "macro_f1_holdout": round(f1_score(train.damage_grade.iloc[idx_hold],
                                                        prediction, average="macro"), 4)})
display(pd.DataFrame(geo_rows).sort_values("macro_f1_holdout", ascending=False))

dominance = train.groupby("ward_id").damage_grade.apply(
    lambda series: series.value_counts(normalize=True).iloc[0])
print(f"Rata-rata proporsi kelas dominan per ward : {dominance.mean():.3f}")
print(f"Ward dengan satu kelas dominan > 70%      : {(dominance > 0.7).mean():.1%}")

top_wards = train.ward_id.value_counts().head(15).index
pivot = pd.crosstab(train.loc[train.ward_id.isin(top_wards), "ward_id"],
                    train.loc[train.ward_id.isin(top_wards), "damage_grade"],
                    normalize="index").reindex(columns=LABELS)
fig, ax = plt.subplots(figsize=(12, 5))
pivot.plot.barh(stacked=True, ax=ax, width=0.8)
ax.set_title("Komposisi tingkat kerusakan pada 15 ward dengan bangunan terbanyak")
ax.set_xlabel("proporsi")
ax.set_ylabel("ward_id")
ax.invert_yaxis()
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
fig.savefig(FIG_DIR / "04_ward_composition.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.5 Struktur `building_id` — lokasi yang lebih halus

`building_id` bukan sekadar nomor acak. Dua digit pertamanya **persis sama dengan `district_id`**,
dan digit-digit berikutnya membentuk kelompok yang jauh lebih halus daripada `ward_id`. Bagian ini
memeriksa strukturnya dan — yang paling penting — memastikan kelompok tersebut **tersedia juga pada
data test**, karena fitur yang hanya ada di train tidak akan menolong di leaderboard.

In [ ]:
bid_train = train.building_id.astype(str)
bid_test = test.building_id.astype(str)

print("Panjang building_id          :", sorted(bid_train.str.len().unique()))
print("2 digit pertama == district_id:",
      f"{(bid_train.str.slice(0, 2).astype(int) == train.district_id).mean():.1%} baris train")

# Setiap segmen 2 digit diperiksa ketersediaannya pada data test.
segment_rows = []
for position in range(1, 6):
    start, stop = 2 * position, 2 * position + 2
    train_segment, test_segment = bid_train.str.slice(start, stop), bid_test.str.slice(start, stop)
    segment_rows.append({
        "segmen": f"bid{position}",
        "n_unik_train": train_segment.nunique(),
        "n_unik_test": test_segment.nunique(),
        "baris_test_nilai_baru (%)": round(100 * (~test_segment.isin(set(train_segment))).mean(), 2),
    })
segment_table = pd.DataFrame(segment_rows)
display(segment_table)

print("Contoh pembacaan (2 digit pertama = district_id):")
display(pd.concat([train[["building_id", "district_id", "vdcmun_id", "ward_id"]].head(5).reset_index(drop=True),
                   pd.DataFrame({f"bid{i}": bid_train.str.slice(2 * i, 2 * i + 2).head(5).reset_index(drop=True)
                                 for i in (1, 2, 3, 4)})], axis=1).head(5))

combined_train = bid_train.str.slice(4, 10)
print(f"Gabungan bid2-bid4: {combined_train.nunique():,} kelompok unik pada train, "
      f"{100 * (~bid_test.str.slice(4, 10).isin(set(combined_train))).mean():.2f}% baris test bernilai baru.")
print("Karena tidak ada nilai baru, segmen ini aman dipakai sebagai fitur kategorikal.")

### 3.6 Ringkasan temuan EDA

1. **Tidak ada nilai hilang** dan **tidak ada kebocoran ID**; `building_id` unik dan tidak
   beririsan antara train dan test. Karena itu validasi *stratified* acak dapat dibenarkan,
   dengan catatan pada Bagian 5.
2. **Kelas tidak seimbang** dengan rasio sekitar 3,5 : 1 (Grade 5 ≈ 36%, Grade 1 ≈ 10%).
   Macro F1 membuat kelas minoritas sama pentingnya, sehingga `class_weight` perlu diuji.
3. **Lokasi adalah fitur dominan.** Aturan "kelas mayoritas per `ward_id`" saja sudah mencapai
   Macro F1 sekitar 0,46 — hampir menyamai seluruh model fitur pada eksperimen awal. Kerusakan
   bersifat sangat spasial: bangunan di ward yang sama cenderung mengalami tingkat kerusakan
   serupa. Sebaliknya, menghapus fitur geografis menjatuhkan performa secara drastis.
4. **Kategori pada test semuanya dikenal** oleh train (tidak ada kategori baru). Meskipun
   demikian, encoder tetap dikonfigurasi agar aman terhadap nilai tak dikenal.
5. **`building_id` menyimpan lokasi yang lebih halus.** Dua digit pertamanya sama dengan
   `district_id`, dan segmen berikutnya membentuk kelompok sub-wilayah. Fitur ini menaikkan
   Macro F1 holdout dari sekitar 0,55 menjadi sekitar 0,57 pada pengujian terpisah, dan seluruh
   nilainya tersedia pada data test sehingga tidak berisiko gagal di leaderboard.
6. `age_building = 999` adalah kode khusus dan perlu indikator tersendiri.

## 4. Feature engineering dan preprocessing

Dua komponen, keduanya berada **di dalam** `Pipeline` sehingga dipelajari ulang pada setiap fold
dan tidak pernah melihat data validasi:

**a. `BuildingFeatures`** — transformasi deterministik tanpa target: tiga ID geografis
diperlakukan sebagai kategori (bukan jarak numerik); **segmen 2 digit `building_id`**
(`bid2`, `bid3`, `bid4`) diambil sebagai kategori tambahan dan kolom `building_id` aslinya dibuang;
lalu diturunkan `height_per_floor`, `estimated_floor_area`, `age_is_999`, dan `log_plinth_area`.
Pembagian dengan nol menghasilkan `NaN` yang diimputasi di dalam fold, bukan diganti angka
spekulatif.

**b. `build_preprocessor`** — `ColumnTransformer` dengan dua cabang: numerik (imputasi median,
opsional penskalaan) dan kategorikal (imputasi modus + encoding). Model linear memakai
*one-hot*; model pohon memakai *ordinal* karena LightGBM dapat memakai pemisahan kategorikal
asli, yang lebih hemat memori untuk `ward_id` dengan 945 level.

In [ ]:
class BuildingFeatures(BaseEstimator, TransformerMixin):
    """Feature engineering deterministik dari data pra-gempa, tanpa informasi target.

    Kelas ini adalah *transformer* scikit-learn sehingga dapat dirangkai di dalam
    ``Pipeline`` dan di-*fit* ulang pada setiap fold validasi.
    """

    ID_COLUMNS = ("district_id", "vdcmun_id", "ward_id")
    # Segmen 2 digit building_id yang dipakai sebagai fitur kategorikal.
    # Dua digit pertama sama dengan district_id sehingga tidak diulang.
    BUILDING_ID_SEGMENTS = (2, 3, 4)

    def fit(self, X: pd.DataFrame, y=None):
        """Catat skema kolom masukan tanpa mempelajari nilai apa pun dari data."""
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Kembalikan fitur turunan dengan urutan baris yang dipertahankan.

        Segmen ``building_id`` harus diambil **sebelum** kolom ID mentahnya dibuang.
        """
        available = [column for column in self.feature_names_in_ if column in X.columns]
        X = X.loc[:, available].copy()

        building_id_text = X["building_id"].astype(str) if "building_id" in X.columns else None
        X = X.drop(columns=["building_id"], errors="ignore")

        # ID lokasi adalah kategori, bukan besaran yang punya jarak numerik.
        for column in self.ID_COLUMNS:
            X[column] = X[column].astype("string").fillna("missing").astype(object)

        floors = X["count_floors_pre_eq"].replace(0, np.nan)
        X["height_per_floor"] = X["height_ft_pre_eq"] / floors
        X["estimated_floor_area"] = X["plinth_area_sq_ft"] * floors
        X["age_is_999"] = X["age_building"].eq(999).astype(int)
        X["log_plinth_area"] = np.log1p(X["plinth_area_sq_ft"].clip(lower=0))

        # Kelompok sub-wilayah dari building_id, mis. "08", "03", "13".
        if building_id_text is not None:
            for position in self.BUILDING_ID_SEGMENTS:
                start, stop = 2 * position, 2 * position + 2
                X[f"bid{position}"] = building_id_text.str.slice(start, stop).astype(object)

        return X.replace([np.inf, -np.inf], np.nan)


# Kolom hasil feature engineering, dipakai untuk menyusun ColumnTransformer.
NUMERIC_FEATURES = NUMERIC_RAW + ["height_per_floor", "estimated_floor_area",
                                  "age_is_999", "log_plinth_area"]
# Segmen building_id diperlakukan sebagai kategorikal (lihat Bagian 3.5).
BUILDING_ID_SEGMENTS = BuildingFeatures.BUILDING_ID_SEGMENTS
SEGMENT_FEATURES = [f"bid{position}" for position in BUILDING_ID_SEGMENTS]
CATEGORICAL_FEATURES = CAT_COLS + SEGMENT_FEATURES

# Indeks kolom kategorikal setelah preprocessing. Urutan keluaran ColumnTransformer
# mengikuti urutan transformer: blok numerik lebih dulu, lalu blok kategorikal.
CATEGORICAL_INDEX = list(range(len(NUMERIC_FEATURES),
                               len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)))
print("Fitur numerik   :", NUMERIC_FEATURES)
print("Fitur kategorikal:", CATEGORICAL_FEATURES)
print("Indeks kategorikal untuk LightGBM:", CATEGORICAL_INDEX)

_smoke = BuildingFeatures().fit_transform(train.head(5))
display(_smoke)

In [ ]:
def build_preprocessor(categorical: list, numeric: list, kind: str = "tree") -> ColumnTransformer:
    """Bangun ``ColumnTransformer`` untuk imputasi dan encoding.

    Parameter
    ----------
    categorical, numeric : list
        Nama kolom untuk masing-masing cabang.
    kind : {"tree", "linear"}
        ``"linear"`` memakai penskalaan dan *one-hot encoding*; ``"tree"`` memakai
        *ordinal encoding* agar LightGBM dapat memakai pemisahan kategorikal asli.
    """
    if kind == "linear":
        numeric_steps = [("imputer", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler())]
        encoder = OneHotEncoder(handle_unknown="ignore")
    else:
        numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                      ("encoder", encoder)]), categorical),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def build_pipeline(kind: str, categorical: list, numeric: list,
                   seed: int = SEED, **model_params) -> Pipeline:
    """Rakit ``Pipeline`` lengkap: feature engineering → preprocessing → model.

    Semua langkah berada dalam satu objek sehingga dapat langsung dipakai pada
    ``cross_val_score`` maupun ``RandomizedSearchCV``.

    Parameter ``model_params`` boleh memakai awalan ``"model__"`` (seperti keluaran
    ``search.best_params_``) maupun tanpa awalan. Kunci yang tidak dikenal akan
    memunculkan ``ValueError``, bukan diabaikan diam-diam.
    """
    models = {
        "dummy": DummyClassifier(strategy="prior"),
        "logistic": LogisticRegression(max_iter=300, class_weight="balanced", random_state=seed),
        "forest": RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=3,
                                         class_weight="balanced_subsample",
                                         n_jobs=-1, random_state=seed),
        "lightgbm": LGBMClassifier(n_estimators=300, learning_rate=0.08, num_leaves=63,
                                   max_bin=127, colsample_bytree=0.9, reg_lambda=2.0,
                                   class_weight="balanced", verbosity=-1,
                                   n_jobs=-1, random_state=seed),
    }
    if kind not in models:
        raise ValueError(f"Jenis model tidak dikenal: {kind}")

    # Terima kunci dengan maupun tanpa awalan "model__", karena
    # ``RandomizedSearchCV.best_params_`` mengembalikan kunci berawalan tersebut.
    cleaned = {key.split("__", 1)[1] if key.startswith("model__") else key: value
               for key, value in model_params.items()}

    # PENTING: LGBMClassifier menerima **kwargs, sehingga kunci yang tidak dikenal
    # diabaikan tanpa error. Tanpa validasi ini, hyperparameter hasil tuning bisa
    # "hilang diam-diam" dan model tetap memakai nilai default sementara laporan
    # mengklaim konfigurasi hasil tuning. Validasi ini menutup celah tersebut.
    unknown = sorted(set(cleaned) - set(models[kind].get_params()))
    if unknown:
        raise ValueError(f"Parameter tidak dikenal untuk {kind}: {unknown}")
    models[kind].set_params(**cleaned)

    preprocessor_kind = "linear" if kind in ("dummy", "logistic") else "tree"
    return Pipeline([
        ("features", BuildingFeatures()),
        ("preprocess", build_preprocessor(categorical, numeric, preprocessor_kind)),
        ("model", models[kind]),
    ])


# Cek cepat: satu model kecil harus bisa dilatih tanpa error sebelum biaya besar dikeluarkan.
_check = build_pipeline("lightgbm", CATEGORICAL_FEATURES, NUMERIC_FEATURES, n_estimators=10)
_check.fit(train.head(2_000), train.damage_grade.head(2_000),
           model__categorical_feature=CATEGORICAL_INDEX)
print("Pipeline berjalan. Bentuk matriks setelah preprocessing:",
      _check.named_steps["preprocess"].transform(
          _check.named_steps["features"].transform(train.head(5))).shape)

## 5. Protokol validasi

Pembagian **stratified 80/20** dengan `random_state=42`: 80% *development* untuk pelatihan dan
penyetelan, 20% *holdout* yang **hanya** disentuh sekali di akhir.

**Apa yang diukur dan apa yang tidak.** Karena `building_id` unik dan tidak ada kelompok
berulang, validasi ini mengukur kemampuan memprediksi **bangunan baru di dalam populasi wilayah
yang sudah dikenal** — persis seperti kondisi test kompetisi, yang seluruh kategori wilayahnya
juga muncul di train. Validasi ini **bukan** bukti kemampuan generalisasi ke wilayah yang belum
pernah dilihat; untuk itu diperlukan validasi *spatial/group* terpisah.

Karena lokasi sangat dominan (Bagian 3.4), perlu ditegaskan: skor tinggi di sini sebagian besar
berasal dari menghafal pola kerusakan per ward, bukan dari pemahaman fisik bangunan.

In [ ]:
X = train.drop(columns="damage_grade")
y = train.damage_grade

X_dev, X_hold, y_dev, y_hold = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

print(f"Development : {len(X_dev):,} baris")
print(f"Holdout     : {len(X_hold):,} baris")

distribution = pd.DataFrame({
    "development (%)": (y_dev.value_counts(normalize=True).reindex(LABELS) * 100).round(2),
    "holdout (%)": (y_hold.value_counts(normalize=True).reindex(LABELS) * 100).round(2),
})
display(distribution)

if QUICK_MODE:
    X_dev, _, y_dev, _ = train_test_split(X_dev, y_dev, train_size=60_000,
                                          stratify=y_dev, random_state=SEED)
    X_hold, _, y_hold, _ = train_test_split(X_hold, y_hold, train_size=20_000,
                                            stratify=y_hold, random_state=SEED)
    print("QUICK_MODE aktif — subset kecil, hasilnya BUKAN hasil kompetisi.")

## 6. Baseline

Tiga baseline pada subset *development* yang sama agar perbandingan adil dan hemat CPU:

| Model | Peran |
|---|---|
| `DummyClassifier(strategy="prior")` | Batas bawah: selalu memprediksi proporsi kelas |
| `Logistic Regression` | Baseline linear, dengan *one-hot* + penskalaan |
| `Random Forest` | Baseline non-linear *bagging* |

Ukuran data latih dicantumkan pada tabel hasil karena baseline dan model final tidak dilatih pada
jumlah baris yang sama; skornya tidak boleh dibandingkan tanpa memperhatikan hal ini.

In [ ]:
def evaluate(pipeline: Pipeline, X_train, y_train, X_eval, y_eval,
             name: str, categorical_index=None) -> dict:
    """Latih pipeline dan kembalikan Macro F1 pada data evaluasi beserta waktu latih."""
    fit_params = {"model__categorical_feature": categorical_index} if categorical_index else {}
    start = time.perf_counter()
    pipeline.fit(X_train, y_train, **fit_params)
    seconds = time.perf_counter() - start
    prediction = pipeline.predict(X_eval)
    return {"model": name,
            "baris_latih": len(X_train),
            "macro_f1_holdout": round(f1_score(y_eval, prediction, average="macro"), 4),
            "detik": round(seconds, 1),
            "prediction": prediction}


BASELINE_ROWS = 20_000 if QUICK_MODE else 120_000
X_base, _, y_base, _ = train_test_split(X_dev, y_dev,
                                        train_size=min(BASELINE_ROWS, len(X_dev)),
                                        stratify=y_dev, random_state=SEED)

baseline_results, baseline_predictions = [], {}
for kind, name in [("dummy", "Dummy (prior)"),
                   ("logistic", "Logistic Regression"),
                   ("forest", "Random Forest")]:
    outcome = evaluate(build_pipeline(kind, CATEGORICAL_FEATURES, NUMERIC_FEATURES),
                       X_base, y_base, X_hold, y_hold, name)
    baseline_predictions[name] = outcome.pop("prediction")
    baseline_results.append(outcome)
    print(f"{name:22s} Macro F1 = {outcome['macro_f1_holdout']:.4f}   ({outcome['detik']} detik)")

display(pd.DataFrame(baseline_results))

## 7. Penyetelan LightGBM

LightGBM dipilih karena dapat menangani fitur kategorikal ber-kardinalitas tinggi secara asli dan
merupakan *gradient boosting* pohon — **bukan** Neural Network, sesuai batasan tugas.

`RandomizedSearchCV` dijalankan **hanya pada data development**. Holdout tidak pernah masuk ke
pencarian. Ruang pencarian mencakup kapasitas pohon (`num_leaves`, `min_child_samples`),
laju belajar, regularisasi, dan **`class_weight`** (`None` vs `balanced`) karena Macro F1
menuntut perhatian khusus pada kelas minoritas.

Ruang pencarian sengaja digeser ke kapasitas yang **lebih besar** daripada versi awal notebook.
Setelah segmen `building_id` ditambahkan, kapasitas ternyata kembali berguna: pengujian terpisah
pada split yang sama memberi 0,5670 (`num_leaves=63`), 0,5723 (`num_leaves=127`), dan 0,5755
(`num_leaves=255`). Karena itu `num_leaves` dicari pada rentang 127–511.

**Perkiraan waktu.** Dengan 10 konfigurasi × 3 fold pada 426.772 baris, bagian ini biasanya
memakan **30–60 menit** di CPU Kaggle. Turunkan `TUNING_ITER` bila ingin lebih cepat.

In [ ]:
TUNING_ITER = 2 if QUICK_MODE else 10
CV_FOLDS = 2 if QUICK_MODE else 3

base_pipeline = build_pipeline("lightgbm", CATEGORICAL_FEATURES, NUMERIC_FEATURES)
if QUICK_MODE:
    base_pipeline.set_params(model__n_estimators=40)

# Rentang kapasitas yang lebih besar, karena segmen building_id membuat model
# mampu memanfaatkan pohon yang lebih dalam.
param_distributions = {
    "model__num_leaves": [127, 255, 383, 511],
    "model__min_child_samples": [10, 20, 40],
    "model__learning_rate": [0.04, 0.06, 0.08],
    "model__n_estimators": [450, 700, 900],
    "model__colsample_bytree": [0.5, 0.7, 0.9],
    "model__reg_lambda": [1.0, 10.0, 30.0],
    "model__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    base_pipeline,
    param_distributions,
    n_iter=TUNING_ITER,
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED),
    random_state=SEED,
    n_jobs=1,
    refit=False,
    verbose=0,
)

start = time.perf_counter()
search.fit(X_dev, y_dev, model__categorical_feature=CATEGORICAL_INDEX)
elapsed = time.perf_counter() - start
print(f"Pencarian selesai dalam {elapsed / 60:.1f} menit "
      f"({TUNING_ITER} konfigurasi x {CV_FOLDS} fold, hanya pada development).")

In [ ]:
cv_table = pd.concat([
    pd.DataFrame([{key.replace("model__", ""): value for key, value in params.items()}
                  for params in search.cv_results_["params"]]),
    pd.DataFrame({"macro_f1_cv": search.cv_results_["mean_test_score"].round(4),
                  "std": search.cv_results_["std_test_score"].round(4)}),
], axis=1).sort_values("macro_f1_cv", ascending=False).reset_index(drop=True)

display(cv_table)
print("Parameter terbaik :", search.best_params_)
print(f"CV Macro F1 terbaik: {search.best_score_:.4f}")
print(f"Sebaran skor antar konfigurasi: {cv_table.macro_f1_cv.min():.4f} - {cv_table.macro_f1_cv.max():.4f}")

best_params = dict(search.best_params_)

## 8. Evaluasi pada holdout

Model dengan parameter terbaik dilatih ulang pada **seluruh** data development, lalu dievaluasi
sekali pada holdout yang belum pernah dilihat.

In [ ]:
class SeedEnsemble:
    """Rata-ratakan probabilitas beberapa ``Pipeline`` dengan hyperparameter identik.

    Rata-rata beberapa seed mengurangi varians prediksi tanpa mengubah arsitektur
    model. Model dasarnya tetap *gradient boosting* pohon, bukan Neural Network.
    """

    def __init__(self, pipelines: list):
        if not pipelines:
            raise ValueError("SeedEnsemble membutuhkan minimal satu pipeline.")
        self.pipelines = pipelines

    def predict_proba(self, X):
        """Kembalikan probabilitas kelas hasil rata-rata seluruh pipeline."""
        return np.mean([pipeline.predict_proba(X) for pipeline in self.pipelines], axis=0)

    def predict(self, X):
        """Kembalikan label kelas berstring sesuai urutan kelas model pertama."""
        classes = self.pipelines[0].named_steps["model"].classes_
        return classes[self.predict_proba(X).argmax(axis=1)]


def fit_seed_ensemble(X_train, y_train, label: str) -> SeedEnsemble:
    """Latih beberapa pipeline ber-seed berbeda dengan hyperparameter terpilih."""
    pipelines = []
    for seed in FINAL_SEEDS:
        started = time.perf_counter()
        pipeline = build_pipeline("lightgbm", CATEGORICAL_FEATURES, NUMERIC_FEATURES,
                                  seed=seed, **best_params)
        pipeline.fit(X_train, y_train, model__categorical_feature=CATEGORICAL_INDEX)
        pipelines.append(pipeline)
        print(f"  {label}: seed {seed} selesai dalam {time.perf_counter() - started:.0f} detik")
    return SeedEnsemble(pipelines)


# Bukti eksplisit bahwa hyperparameter hasil tuning benar-benar terpasang, bukan
# hanya tercatat di laporan. LightGBM mengabaikan kunci tak dikenal tanpa error,
# sehingga pengecekan ini penting.
_probe = build_pipeline("lightgbm", CATEGORICAL_FEATURES, NUMERIC_FEATURES, **best_params)
_fitted_params = _probe.named_steps["model"].get_params()
applied_params = {key.replace("model__", ""): _fitted_params[key.replace("model__", "")]
                  for key in sorted(best_params)}
print("Hyperparameter yang benar-benar terpasang pada estimator final:")
for _name, _value in applied_params.items():
    _expected = best_params[f"model__{_name}"]
    print(f"  {_name:20s} = {_value!r:20s} (diminta: {_expected!r})")
    assert _value == _expected or str(_value) == str(_expected), f"{_name} tidak terpasang!"
print("Seluruh hyperparameter hasil tuning terverifikasi terpasang.")
del _probe

print(f"Melatih {len(FINAL_SEEDS)} seed pada development ({len(X_dev):,} baris)...")
dev_ensemble = fit_seed_ensemble(X_dev, y_dev, "development")

holdout_prediction = dev_ensemble.predict(X_hold)
holdout_f1 = f1_score(y_hold, holdout_prediction, average="macro")
print(f"\n>>> HOLDOUT MACRO F1 = {holdout_f1:.4f} (rata-rata {len(FINAL_SEEDS)} seed)")

report = classification_report(y_hold, holdout_prediction, labels=LABELS,
                               output_dict=True, zero_division=0)
report_table = pd.DataFrame(report).T.loc[LABELS + ["macro avg", "weighted avg"],
                                          ["precision", "recall", "f1-score", "support"]]
display(report_table.round(4))

### 8.1 Confusion matrix — kelas mana yang tertukar?

Karena Macro F1 menimbang setiap kelas setara, kelas dengan recall rendah adalah sumber utama
kehilangan skor.

In [ ]:
cm = confusion_matrix(y_hold, holdout_prediction, labels=LABELS)
cm_normalized = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion matrix (jumlah bangunan)")
ConfusionMatrixDisplay(cm_normalized, display_labels=LABELS).plot(
    ax=axes[1], cmap="Greens", colorbar=False, values_format=".2f")
axes[1].set_title("Confusion matrix (proporsi per kelas aktual)")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
fig.savefig(FIG_DIR / "05_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

recalls = {label: report[label]["recall"] for label in LABELS}
print("Recall per kelas (rendah → tinggi):")
for label, value in sorted(recalls.items(), key=lambda item: item[1]):
    print(f"  {label}: {value:.3f}")

off_diagonal = sorted(
    ((LABELS[i], LABELS[j], int(cm[i, j]))
     for i in range(len(LABELS)) for j in range(len(LABELS)) if i != j),
    key=lambda row: -row[2])
print("\nKesalahan terbanyak (aktual → prediksi):")
for true_label, predicted_label, count in off_diagonal[:5]:
    print(f"  {true_label} → {predicted_label}: {count:,} bangunan")

### 8.2 Permutation importance

Setiap fitur diacak dan penurunan Macro F1 diukur pada subset holdout. Metode ini mengukur
**seberapa besar model bergantung pada fitur tersebut**, bukan hubungan sebab-akibat.

`building_id` sengaja diikutsertakan sebagai kontrol: karena dibuang oleh `BuildingFeatures`,
angkanya harus mendekati nol. Jika tidak, ada indikasi kebocoran.

In [ ]:
IMPORTANCE_ROWS = 500 if QUICK_MODE else 3_000
X_imp_raw, _, y_imp, _ = train_test_split(X_hold, y_hold, train_size=IMPORTANCE_ROWS,
                                          stratify=y_hold, random_state=SEED)

# Importance dihitung pada **fitur hasil rekayasa**, bukan kolom mentah, agar
# kontribusi ward_id dan tiap segmen building_id (bid2, bid3, bid4) terlihat
# terpisah. Model yang dipakai adalah tahap setelah feature engineering, sehingga
# kolom building_id mentah tidak lagi ikut diacak.
X_imp = BuildingFeatures().fit_transform(X_imp_raw)
downstream = Pipeline([("preprocess", dev_ensemble.pipelines[0].named_steps["preprocess"]),
                       ("model", dev_ensemble.pipelines[0].named_steps["model"])])

started = time.perf_counter()
importance = permutation_importance(downstream, X_imp, y_imp, scoring="f1_macro",
                                    n_repeats=2 if QUICK_MODE else 3,
                                    random_state=SEED, n_jobs=1)
print(f"Perhitungan importance selesai dalam {time.perf_counter() - started:.0f} detik "
      f"pada {len(X_imp):,} baris holdout")

importance_table = pd.DataFrame({
    "fitur": X_imp.columns,
    "penurunan_macro_f1": importance.importances_mean,
    "std": importance.importances_std,
}).sort_values("penurunan_macro_f1", ascending=False).reset_index(drop=True)
display(importance_table.round(4))

top = importance_table.head(12).sort_values("penurunan_macro_f1")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.fitur, top.penurunan_macro_f1, xerr=top["std"], color="#55A868")
ax.set_xlabel("Penurunan Macro F1 saat fitur diacak")
ax.set_title("Permutation importance pada holdout")
fig.savefig(FIG_DIR / "06_permutation_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Fitur paling berpengaruh:", ", ".join(importance_table.fitur.head(5)))

### 8.3 Interpretasi hasil evaluasi

Bagian ini merangkum cara membaca keluaran di atas. Angka pastinya berasal dari eksekusi nyata
notebook ini, bukan nilai yang dituliskan sebelumnya — lihat `reports/results_v1.json`.

**Performa.** Bandingkan `macro avg` f1-score pada tabel classification report dengan baseline
di Bagian 6. Selisih terhadap `Dummy (prior)` menunjukkan nilai tambah pemodelan, sedangkan
selisih terhadap `Logistic Regression` menunjukkan manfaat interaksi non-linear.

**Kelas minoritas adalah penentu Macro F1.** Karena jumlah bangunan Grade 1 paling sedikit,
kontribusinya terhadap Macro F1 sama besar dengan Grade 5. Perhatikan recall tiap kelas:
biasanya kelas minoritas (Grade 1–2) memiliki recall lebih rendah, dan justru di situ skor
paling banyak hilang.

**Kelas yang sering tertukar.** Confusion matrix umumnya menunjukkan kesalahan bersifat
*berdekatan*: Grade 1 tertukar dengan Grade 2, dan Grade 4 dengan Grade 5, karena tingkat
kerusakan bersifat ordinal. Kesalahan Grade 3 → Grade 4 biasanya paling banyak secara absolut
karena Grade 3 adalah kelas "transisi". Kesalahan yang jauh (mis. Grade 1 → Grade 5) jarang
terjadi dan menandakan bangunan dengan kombinasi struktur yang tidak lazim di ward-nya.

**Fitur berpengaruh.** Permutation importance konsisten dengan temuan EDA: fitur geografis
mendominasi, diikuti material struktur (`superstructure`, `foundation_type`) dan rasio geometri
bangunan. Perhatikan juga posisi `bid2`/`bid3`/`bid4` — segmen `building_id` yang menandai
kelompok sub-wilayah. Fitur inilah yang menaikkan skor paling besar dibandingkan versi awal
notebook, dan karena seluruh nilainya tersedia pada data test (Bagian 3.5), kenaikannya
seharusnya terbawa ke leaderboard.

**Bagaimana `building_id` boleh dipakai?** Kolom ini tersedia pada train **dan** test, tidak
diturunkan dari target, dan bukan pengenal unik bangunan yang bocor. Isinya paling masuk akal
sebagai kode blok survei/enumerasi: tingkat kerusakan memang terkonsentrasi sampai skala
sub-wilayah. Karena itu pemakaiannya sah, tetapi **interpretasinya harus hati-hati**: sebagian
sinyalnya bisa mencerminkan cara tim survei bekerja, bukan semata kondisi fisik bangunan.

**Risiko generalisasi.** Karena sinyal geografis begitu kuat, angka holdout ini
**mengasumsikan wilayah test sudah dikenal**, yang memang benar untuk kompetisi ini. Model yang
sama tidak dapat dipakai untuk memprediksi distrik yang belum pernah disurvei.

## 9. Pelatihan final dan submission

Konfigurasi terbaik dilatih ulang pada **seluruh 533.465 baris** train (development + holdout)
untuk memaksimalkan data, dan diulang pada beberapa seed lalu probabilitasnya dirata-ratakan,
lalu dipakai memprediksi test. Validasi submission bersifat ketat: ID
dicocokkan lewat *mapping* `building_id` (bukan mengasumsikan urutan baris), label harus berada
pada rentang yang sah, dan CSV dibaca ulang untuk memastikan isi tersimpan benar.

**Format label.** Kolom `damage_grade` ditulis sebagai **bilangan bulat 1–5**, bukan string
`Grade N`. Alasannya: skrip penilai kompetisi mem-*parse* kolom ini sebagai angka, terbukti dari
*Evaluation Exception* yang muncul saat submission berisi `Grade 1`. Gaya label diatur oleh
`SUBMISSION_LABEL_STYLE` di bagian Setup; nilai `"grade"` akan menghasilkan
`submission_grade_labels.csv` dengan string asli bila sewaktu-waktu diperlukan. Verifikasi tipe
data dilakukan pada berkas yang benar-benar tersimpan di disk, bukan hanya pada DataFrame di
memori.

In [ ]:
def make_submission(model, test_frame: pd.DataFrame, sample_frame: pd.DataFrame,
                    output_path: Path, label_style: str = "numeric") -> pd.DataFrame:
    """Prediksi, selaraskan ke ID sample, validasi, lalu simpan berkas submission.

    Parameter
    ----------
    label_style : {"numeric", "grade"}
        ``"numeric"`` menulis ``damage_grade`` sebagai bilangan bulat 1-5 (dipakai
        untuk submission), ``"grade"`` menulis string ``"Grade 1"``-``"Grade 5"``
        seperti pada ``sample_submission.csv``.

    Selaraskan lewat *mapping* ``building_id`` agar urutan baris test tidak menjadi
    asumsi yang rapuh.
    """
    if label_style not in ("numeric", "grade"):
        raise ValueError("label_style harus 'numeric' atau 'grade'.")
    if list(sample_frame.columns) != ["building_id", "damage_grade"]:
        raise ValueError("Kolom sample_submission tidak sesuai.")
    if len(sample_frame) != len(test_frame):
        raise ValueError("Jumlah baris test dan sample_submission berbeda.")
    if not test_frame.building_id.is_unique or not sample_frame.building_id.is_unique:
        raise ValueError("building_id harus unik.")
    if set(sample_frame.building_id) != set(test_frame.building_id):
        raise ValueError("Himpunan building_id test dan sample_submission tidak sama.")

    predicted_labels = pd.Series(model.predict(test_frame), index=test_frame.building_id)
    aligned = sample_frame.building_id.map(predicted_labels)

    if aligned.isna().any():
        raise ValueError("Ada bangunan yang belum terprediksi.")
    if not aligned.isin(LABELS).all():
        raise ValueError("Ada label di luar Grade 1-Grade 5.")

    submission = sample_frame.copy()
    if label_style == "numeric":
        submission["damage_grade"] = aligned.map(LABEL_TO_CODE).astype("int64")
    else:
        submission["damage_grade"] = aligned

    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(output_path, index=False)

    reread = pd.read_csv(output_path)
    pd.testing.assert_frame_equal(submission.reset_index(drop=True), reread)
    return submission


SUBMISSION_PATH = OUTPUT_DIR / ("submission_quicktest.csv" if QUICK_MODE else "submission.csv")

started = time.perf_counter()
if FULL_REFIT:
    print(f"Melatih {len(FINAL_SEEDS)} seed pada seluruh {len(X):,} baris train...")
    deployment_model = fit_seed_ensemble(X, y, "seluruh train")
    print(f"Model final siap dalam {time.perf_counter() - started:.0f} detik")
else:
    deployment_model = dev_ensemble
    print("FULL_REFIT=False — memakai model development (hanya untuk uji cepat).")

submission = make_submission(deployment_model, test, sample, SUBMISSION_PATH,
                             label_style=SUBMISSION_LABEL_STYLE)

# Salinan cadangan bergaya "Grade N", disiapkan bila penyelenggara memintanya.
SUBMISSION_GRADE_PATH = SUBMISSION_PATH.with_name(
    "submission_quicktest_grade_labels.csv" if QUICK_MODE else "submission_grade_labels.csv")
make_submission(deployment_model, test, sample, SUBMISSION_GRADE_PATH, label_style="grade")

# Versi label string, dipakai untuk ringkasan laporan agar per kelas mudah dibaca.
submission_as_labels = (submission.damage_grade.map(CODE_TO_LABEL)
                        if SUBMISSION_LABEL_STYLE == "numeric" else submission.damage_grade)

print("Submission tersimpan  :", SUBMISSION_PATH, f"| gaya label: {SUBMISSION_LABEL_STYLE}")
print("Cadangan label string :", SUBMISSION_GRADE_PATH)
print("Bentuk                :", submission.shape)
print("Kolom                 :", list(submission.columns))
print("Tipe damage_grade     :", submission.damage_grade.dtype)
display(submission.head())
display(submission_as_labels.value_counts().reindex(LABELS).to_frame("jumlah prediksi"))

In [ ]:
# Simpan model dan pastikan hasil prediksi tidak berubah setelah dimuat ulang.
MODEL_PATH = OUTPUT_DIR / "models" / "pipeline_model_lightgbm.pkl"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(deployment_model, MODEL_PATH)

restored = joblib.load(MODEL_PATH)
assert np.array_equal(deployment_model.predict(test.head(20)), restored.predict(test.head(20)))
print("Model tersimpan dan konsisten setelah dimuat ulang:", MODEL_PATH)

# Verifikasi format akhir berkas submission yang benar-benar tersimpan di disk.
saved = pd.read_csv(SUBMISSION_PATH)
assert list(saved.columns) == ["building_id", "damage_grade"], "Kolom submission salah."
assert saved.building_id.equals(sample.building_id), "Urutan/isi building_id tidak sesuai sample."
if SUBMISSION_LABEL_STYLE == "numeric":
    assert saved.damage_grade.dtype.kind in "iu", "damage_grade harus bilangan bulat."
    assert set(saved.damage_grade.unique()) <= set(LABEL_TO_CODE.values()), "Kode kelas harus 1-5."
    print("Format terverifikasi: damage_grade bilangan bulat 1-5, tanpa desimal dan tanpa teks.")
else:
    assert saved.damage_grade.isin(LABELS).all(), "Label harus Grade 1-Grade 5."
    print("Format terverifikasi: damage_grade berupa string Grade 1-Grade 5.")

# Perbandingan distribusi prediksi holdout vs test — deteksi pergeseran yang mencolok.
shift_check = pd.DataFrame({
    "holdout aktual (%)": (y_hold.value_counts(normalize=True).reindex(LABELS) * 100).round(2),
    "holdout prediksi (%)": (pd.Series(holdout_prediction).value_counts(normalize=True)
                             .reindex(LABELS).fillna(0) * 100).round(2),
    "test prediksi (%)": (submission_as_labels.value_counts(normalize=True)
                          .reindex(LABELS).fillna(0) * 100).round(2),
})
display(shift_check)

In [ ]:
# Simpan seluruh artefak laporan agar dapat dirujuk pada README/laporan.
REPORT_DIR = OUTPUT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
cv_table.to_csv(REPORT_DIR / "cv_results.csv", index=False)
report_table.to_csv(REPORT_DIR / "classification_report.csv")
importance_table.to_csv(REPORT_DIR / "permutation_importance.csv", index=False)
pd.DataFrame(baseline_results).to_csv(REPORT_DIR / "model_comparison.csv", index=False)
audit.to_csv(REPORT_DIR / "data_audit.csv", index=False)

summary = {
    "competition": "structural-damage-classification",
    "metric": "macro_f1",
    "quick_mode": QUICK_MODE,
    "full_refit": FULL_REFIT,
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "development_rows": int(len(X_dev)),
    "holdout_rows": int(len(X_hold)),
    "baselines": baseline_results,
    "cv_best_macro_f1": round(float(search.best_score_), 4),
    "best_params": {key.replace("model__", ""): value for key, value in best_params.items()},
    "holdout_macro_f1": round(float(holdout_f1), 4),
    "holdout_f1_per_class": {label: round(report[label]["f1-score"], 4) for label in LABELS},
    "holdout_recall_per_class": {label: round(report[label]["recall"], 4) for label in LABELS},
    "submission_label_style": SUBMISSION_LABEL_STYLE,
    "prediction_distribution": {label: int(value) for label, value in
                                submission_as_labels.value_counts().reindex(LABELS).items()},
    "public_score": None,
    "private_score": None,
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "scikit_learn": sklearn.__version__,
                 "lightgbm": lightgbm.__version__},
}
(REPORT_DIR / "results.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 10. Kesimpulan dan keterbatasan

**Yang dikerjakan.** Seluruh preprocessing dan pemodelan dirangkai dalam satu `Pipeline` dengan
`ColumnTransformer`, sehingga imputasi, encoding, dan penskalaan selalu dipelajari ulang di dalam
fold dan tidak pernah melihat data validasi. Tidak ada Neural Network yang dipakai. Tiga baseline
dibandingkan, LightGBM disetel dengan `RandomizedSearchCV` hanya pada data development, dan
holdout dievaluasi satu kali.

**Temuan utama.**

1. **Lokasi mendominasi.** Aturan naif "kelas mayoritas per `ward_id`" saja sudah mencapai
   Macro F1 sekitar 0,46, dan menghapus fitur geografis menjatuhkan model ke sekitar 0,38.
   Kerusakan akibat gempa 2015 terkonsentrasi secara spasial: bangunan di ward yang sama
   cenderung rusak pada tingkat yang sama.
2. **`building_id` menyimpan lokasi sub-wilayah, dan itu adalah pengungkit terbesar.**
   Dua digit pertamanya sama dengan `district_id`; segmen berikutnya menandai kelompok yang
   lebih halus. Menambahkannya sebagai fitur kategorikal menaikkan Macro F1 holdout dari
   **0,548** menjadi **0,576** pada pengujian terpisah dengan split yang sama. Peningkatan ini
   tetap berlaku karena **0,00% baris test** memiliki nilai segmen yang belum dikenal train.
3. **Kapasitas model berguna setelah fiturnya diperkaya.** Dengan fitur lama, menambah
   `num_leaves` hampir tidak mengubah skor. Setelah segmen `building_id` masuk, kapasitas justru
   berpengaruh (0,5670 → 0,5723 → 0,5755 untuk `num_leaves` 63 → 127 → 255). Ini contoh klasik
   bahwa *feature engineering* menentukan seberapa besar kapasitas model bisa dimanfaatkan.
4. **Jumlah data latih sangat menentukan.** Pelatihan pada 120 ribu baris menghasilkan skor jauh
   lebih rendah daripada 426 ribu baris, sehingga model final dilatih pada seluruh 533 ribu baris.
5. **`class_weight="balanced"` membantu** karena Macro F1 menimbang kelas minoritas setara
   dengan kelas mayoritas. Sebaliknya, koreksi prior pasca-prediksi (*logit adjustment*) tidak
   memberi tambahan setelah pembobotan diterapkan.
6. **Yang tidak membantu (sudah diuji).** *Target encoding* untuk `ward_id`, penyetelan
   `cat_smooth`/`cat_l2`/`max_cat_threshold`, objective *One-vs-All* (`multiclassova`), agregat
   statistik per ward, dan CatBoost (0,535 — lebih rendah dan jauh lebih lambat). Pemisahan
   kategorikal asli LightGBM sudah cukup efisien untuk `ward_id` dengan 945 level.

**Keterbatasan yang harus disebutkan dalam laporan.**

- Validasi *stratified* acak mengukur bangunan baru di **wilayah yang sudah dikenal**. Ini sesuai
  kondisi test kompetisi, tetapi **bukan** bukti generalisasi ke wilayah baru. Validasi
  *spatial/group* (mis. `GroupKFold` per `ward_id`) akan memberi estimasi yang jauh lebih
  rendah dan lebih jujur untuk klaim tersebut.
- Terdapat ribuan baris dengan kombinasi fitur identik; ini bukan bukti bangunan yang sama,
  tetapi menunjukkan keterbatasan resolusi fitur. Dua bangunan dengan fitur identik bisa
  memiliki tingkat kerusakan berbeda, sehingga ada batas atas akurasi yang tidak bisa dilewati.
- `age_building = 999` dan pentingnya arti sebenarnya belum terkonfirmasi; nilainya dipertahankan
  dan hanya ditandai, tidak diganti.
- Kolom `superstructure` hanya berisi satu label, sedangkan `metadata.csv` menyebut 11 *flag*
  material yang bisa saja muncul bersamaan. Informasi multi-material yang hilang ini membatasi
  performa dan tidak dapat dipulihkan dari CSV yang tersedia.
- `public_score` dan `private_score` pada `reports/results_v1.json` masih `null` sampai submission
  benar-benar dikirim ke Kaggle. Skor holdout lokal **tidak boleh** disamakan dengan public score.

**Langkah berikutnya bila ingin menaikkan skor.** (1) Validasi `GroupKFold` per `ward_id` untuk
estimasi yang jujur sekaligus memastikan fitur `building_id` tidak sekadar menghafal kelompok
survei. (2) *Early stopping* per fold agar kapasitas optimal ditemukan tanpa menebak. (3)
*Stacking* ringan dengan Logistic Regression sebagai meta-model di atas probabilitas beberapa
model. (4) Menambah konteks luar data survei, misalnya jarak ke episentrum — sinyal geografis
yang dominan menunjukkan di situlah sisa perbaikan terbesar masih tersimpan.